# HESE Event Comparison

1. Load all events from the per-year EvtGen.h5 files.
2. Compare run coverage against the L5 i3 files from eyildizci's processing.
3. Compare run coverage against the HESE12 i3 files.
4. Apply `reco_energy > 60 TeV` cut (stored separately as `df_evtgen_cut`).
5. Load reference events from HESE12.hdf5.
6. Verify the topology sub-files add up to HESE12.hdf5.
7. Compare EvtGen (after cut) against HESE12.

In [84]:
import re
import tables
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Collect events from per-year EvtGen files

In [85]:
BASE_PATH = "/data/user/tvaneede/GlobalFit/reco_processing/data/hese/output/v2"

DATASETS = [
    "IC79_2010",
    "IC86_2011",
    "IC86_2012",
    "IC86_2013",
    "IC86_2014",
    "IC86_2015",
    # "IC86_2016",
    "IC86_2017",
    "IC86_2018",
    "IC86_2019",
    "IC86_2020",
    "IC86_2021",
    "IC86_2022",
]

In [92]:
records = []

for dataset in DATASETS:
    path = f"{BASE_PATH}/{dataset}/EvtGen/EvtGen.h5"
    with tables.open_file(path, "r") as f:
        df_hdr = pd.DataFrame({
            "run":   f.root.I3EventHeader.col("Run"),
            "event": f.root.I3EventHeader.col("Event"),
        })
        df_en = pd.DataFrame({
            "run":         f.root.RecoETot.col("Run"),
            "event":       f.root.RecoETot.col("Event"),
            "reco_energy": f.root.RecoETot.col("value"),
            "qtot": f.root.HESE_CausalQTot.col("value"),
        })
    df = df_hdr.merge(df_en, on=["run", "event"], how="left")
    df.insert(0, "dataset", dataset)
    records.append(df)
    print(f"{dataset}: {len(df)} events")

df_evtgen = pd.concat(records, ignore_index=True)
print(f"\nTotal events: {len(df_evtgen)}")
df_evtgen[ df_evtgen["dataset"] == "IC86_2011" ]

IC79_2010: 7 events
IC86_2011: 21 events
IC86_2012: 9 events
IC86_2013: 16 events
IC86_2014: 15 events
IC86_2015: 9 events
IC86_2017: 18 events
IC86_2018: 16 events
IC86_2019: 19 events
IC86_2020: 10 events
IC86_2021: 12 events
IC86_2022: 19 events

Total events: 171


,dataset,run,event,reco_energy,qtot
7,IC86_2011,119474,33152537,2.829383e+04,6150.662853
8,IC86_2011,119311,430943,3.269002e+05,14156.095864
9,IC86_2011,118283,9445773,8.295762e+04,7941.826037
10,IC86_2011,119214,8606380,6.528356e+04,7528.893319
11,IC86_2011,119583,141609,5.709847e+04,6875.744233
12,IC86_2011,119404,80750561,2.171546e+05,22020.592094
13,IC86_2011,118549,11722208,5.100263e+04,8657.278915
14,IC86_2011,119674,8449256,1.660964e+05,19902.005564
15,IC86_2011,118381,19162840,8.930833e+04,6618.705410
16,IC86_2011,120045,22615214,5.016608e+04,7428.785810


## 2. Compare EvtGen run coverage with L5 processing (eyildizci)

In [87]:
L5_BASE = "/data/user/eyildizci/hese_tracks_processing/L5"
L5_YEARS = [str(y) for y in range(2011, 2023)]

l5_records = []
for year in L5_YEARS:
    folder = Path(L5_BASE) / f"new_IC86_{year}"
    for fname in sorted(folder.iterdir()):
        m = re.search(r"Run(\d+)", fname.name)
        if m:
            l5_records.append({"dataset": f"IC86_{year}", "run": int(m.group(1))})

df_l5 = pd.DataFrame(l5_records)
print(f"L5 files (runs): {len(df_l5)}")
df_l5.groupby("dataset").size().rename("n_runs")

df_l5[df_l5["dataset"] == "IC86_2014"]

L5 files (runs): 179


,dataset,run
42,IC86_2014,124852
43,IC86_2014,124927
44,IC86_2014,125071
45,IC86_2014,125335
46,IC86_2014,125365
47,IC86_2014,125627
48,IC86_2014,125914
49,IC86_2014,125975
50,IC86_2014,125979
51,IC86_2014,126090


In [93]:
# Compare at run level: EvtGen unique runs vs L5 runs
evtgen_runs = set(zip(df_evtgen["dataset"], df_evtgen["run"]))
l5_runs     = set(zip(df_l5["dataset"],     df_l5["run"]))

only_evtgen = evtgen_runs - l5_runs
only_l5     = l5_runs - evtgen_runs

print(f"Runs in EvtGen also in L5:      {len(evtgen_runs - only_evtgen)} / {len(evtgen_runs)}")
print(f"Runs in EvtGen but NOT in L5:   {len(only_evtgen)}")
print(f"Runs in L5 but NOT in EvtGen:   {len(only_l5)}")

Runs in EvtGen also in L5:      155 / 169
Runs in EvtGen but NOT in L5:   14
Runs in L5 but NOT in EvtGen:   24


In [95]:
# EvtGen events whose run is not in L5
df_evtgen["run_in_l5"] = [
    (ds, r) in l5_runs
    for ds, r in zip(df_evtgen["dataset"], df_evtgen["run"])
]

df_not_in_l5 = (
    df_evtgen[~df_evtgen["run_in_l5"]]
    [["dataset", "run", "event", "reco_energy", "qtot"]]
    .reset_index(drop=True)
)
print(f"EvtGen events whose run is not in L5: {len(df_not_in_l5)}")
df_not_in_l5

EvtGen events whose run is not in L5: 15


,dataset,run,event,reco_energy,qtot
0,IC79_2010,116528,52433389,6.852011e+04,8442.636503
1,IC79_2010,117371,31623515,3.036372e+04,8233.844019
2,IC79_2010,118145,5142726,5.864218e+04,6282.660629
3,IC79_2010,116698,10198436,1.488022e+05,11066.137793
4,IC79_2010,115994,2538090,4.492534e+04,6105.663898
5,IC79_2010,115994,29874216,1.024323e+05,12596.696886
6,IC79_2010,117782,49441871,2.355923e+04,6438.907011
7,IC86_2011,119311,430943,3.269002e+05,14156.095864
8,IC86_2011,119583,141609,5.709847e+04,6875.744233
9,IC86_2012,121947,7181486,7.449678e+04,9355.539456


In [91]:
# L5 runs not represented in EvtGen
df_l5["in_evtgen"] = [(ds, r) in evtgen_runs for ds, r in zip(df_l5["dataset"], df_l5["run"])]

df_l5_missing = df_l5[~df_l5["in_evtgen"]].reset_index(drop=True)
print(f"L5 runs not in EvtGen: {len(df_l5_missing)}")
df_l5_missing

L5 runs not in EvtGen: 24


,dataset,run,in_evtgen
0,IC86_2014,125627,False
1,IC86_2014,125914,False
2,IC86_2016,128027,False
3,IC86_2016,128224,False
4,IC86_2016,128239,False
5,IC86_2016,128290,False
6,IC86_2016,128340,False
7,IC86_2016,128557,False
8,IC86_2016,128582,False
9,IC86_2016,128619,False


## 3. Compare EvtGen run coverage with HESE12 i3 files

In [46]:
HESE12_I3_BASE = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/i3files/NoDeepCore/HESE12/Bfr"

i3_records = []
for dataset in DATASETS:
    if dataset == "IC86_2022": continue
    folder = Path(HESE12_I3_BASE) / dataset
    seen_runs = set()
    for fname in sorted(folder.iterdir()):
        m = re.search(r"Run(\d+)", fname.name)
        if m:
            run = int(m.group(1))
            if run not in seen_runs:
                seen_runs.add(run)
                i3_records.append({"dataset": dataset, "run": run})

df_hese12_i3 = pd.DataFrame(i3_records)
print(f"HESE12 i3 unique runs: {len(df_hese12_i3)}")
df_hese12_i3.groupby("dataset").size().rename("n_runs")

HESE12 i3 unique runs: 141


dataset
IC79_2010     6
IC86_2011    19
IC86_2012     8
IC86_2013    15
IC86_2014    14
IC86_2015     8
IC86_2017    18
IC86_2018    14
IC86_2019    18
IC86_2020     9
IC86_2021    12
Name: n_runs, dtype: int64

In [47]:
# Compare at run level: EvtGen unique runs vs HESE12 i3 runs
hese12_i3_runs = set(zip(df_hese12_i3["dataset"], df_hese12_i3["run"]))

only_evtgen_i3 = evtgen_runs - hese12_i3_runs
only_i3        = hese12_i3_runs - evtgen_runs

print(f"Runs in EvtGen also in HESE12 i3:    {len(evtgen_runs - only_evtgen_i3)} / {len(evtgen_runs)}")
print(f"Runs in EvtGen but NOT in HESE12 i3: {len(only_evtgen_i3)}")
print(f"Runs in HESE12 i3 but NOT in EvtGen: {len(only_i3)}")

# EvtGen events whose run is not in the HESE12 i3 files
df_evtgen["run_in_hese12_i3"] = [
    (ds, r) in hese12_i3_runs
    for ds, r in zip(df_evtgen["dataset"], df_evtgen["run"])
]

df_not_in_hese12_i3 = (
    df_evtgen[~df_evtgen["run_in_hese12_i3"]]
    [["dataset", "run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"\nEvtGen events whose run is not in HESE12 i3 files: {len(df_not_in_hese12_i3)}")
display(df_not_in_hese12_i3)

# HESE12 i3 runs not represented in EvtGen
df_hese12_i3["in_evtgen"] = [(ds, r) in evtgen_runs for ds, r in zip(df_hese12_i3["dataset"], df_hese12_i3["run"])]
df_i3_missing = df_hese12_i3[~df_hese12_i3["in_evtgen"]].reset_index(drop=True)
print(f"\nHESE12 i3 runs not in EvtGen: {len(df_i3_missing)}")
display(df_i3_missing)

Runs in EvtGen also in HESE12 i3:    140 / 169
Runs in EvtGen but NOT in HESE12 i3: 29
Runs in HESE12 i3 but NOT in EvtGen: 1

EvtGen events whose run is not in HESE12 i3 files: 29


,dataset,run,event,reco_energy
0,IC86_2011,119311,430943,3.269002e+05
1,IC86_2011,119583,141609,5.709847e+04
2,IC86_2012,121947,7181486,7.449678e+04
3,IC86_2013,123770,442256,4.169044e+06
4,IC86_2014,126359,9400616,2.526825e+04
5,IC86_2014,125826,470241,4.164990e+05
6,IC86_2015,127751,927145,2.326454e+05
7,IC86_2018,131680,66412090,2.252563e+05
8,IC86_2018,132143,36142391,1.993608e+05
9,IC86_2020,134994,1103075,3.565653e+05



HESE12 i3 runs not in EvtGen: 1


,dataset,run,in_evtgen
0,IC86_2014,125914,False


## 4. Apply reco_energy > 60 TeV cut

In [58]:
ENERGY_CUT = 60e3  # GeV

df_evtgen_cut = df_evtgen[df_evtgen["reco_energy"] > ENERGY_CUT].reset_index(drop=True)
print(f"Events before cut: {len(df_evtgen)}")
print(f"Events after cut:  {len(df_evtgen_cut)}")
df_evtgen_cut.groupby("dataset").size().rename("n_events")

Events before cut: 171
Events after cut:  102


dataset
IC79_2010     3
IC86_2011    12
IC86_2012     4
IC86_2013    12
IC86_2014    11
IC86_2015     7
IC86_2017    12
IC86_2018     7
IC86_2019     8
IC86_2020     7
IC86_2021     7
IC86_2022    12
Name: n_events, dtype: int64

## 5. Load reference events from HESE12.hdf5

In [59]:
HESE12_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr/HESE12.hdf5"

with tables.open_file(HESE12_PATH, "r") as f:
    df_hdr_ref = pd.DataFrame({
        "run":   f.root.I3EventHeader.col("Run"),
        "event": f.root.I3EventHeader.col("Event"),
    })
    df_en_ref = pd.DataFrame({
        "run":         f.root.RecoETot.col("Run"),
        "event":       f.root.RecoETot.col("Event"),
        "reco_energy": f.root.RecoETot.col("value"),
    })

df_hese12 = df_hdr_ref.merge(df_en_ref, on=["run", "event"], how="left")
print(f"HESE12 events: {len(df_hese12)}")
df_hese12.head()

HESE12 events: 97


,run,event,reco_energy
0,134912,80902176,106029.802180
1,131502,58093123,401730.596192
2,131505,52347097,150473.232499
3,129253,4711217,259569.685977
4,128619,66564632,151213.222561


## 6. Check topology sub-files add up to HESE12.hdf5

In [50]:
BFR_PATH = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/data/Pass2/hdf_files/NoDeepCore/HESE12/Bfr"

TOPOLOGY_FILES = {
    "Cascades":       f"{BFR_PATH}/HESE12_Cascades.hdf5",
    "DoubleCascades": f"{BFR_PATH}/HESE12_DoubleCascades.hdf5",
    "Tracks":         f"{BFR_PATH}/HESE12_Tracks.hdf5",
}

topo_dfs = {}
for topology, path in TOPOLOGY_FILES.items():
    with tables.open_file(path, "r") as f:
        df_hdr = pd.DataFrame({
            "run":   f.root.I3EventHeader.col("Run"),
            "event": f.root.I3EventHeader.col("Event"),
        })
        df_en = pd.DataFrame({
            "run":         f.root.RecoETot.col("Run"),
            "event":       f.root.RecoETot.col("Event"),
            "reco_energy": f.root.RecoETot.col("value"),
        })
    df = df_hdr.merge(df_en, on=["run", "event"], how="left")
    df.insert(0, "topology", topology)
    topo_dfs[topology] = df
    print(f"{topology}: {len(df)} events")

df_topo = pd.concat(topo_dfs.values(), ignore_index=True)
print(f"\nTotal across topology files: {len(df_topo)}")
print(f"Total in HESE12.hdf5:        {len(df_hese12)}")

Cascades: 64 events
DoubleCascades: 5 events
Tracks: 28 events

Total across topology files: 97
Total in HESE12.hdf5:        97


In [51]:
topo_set   = set(zip(df_topo["run"],   df_topo["event"]))
hese12_set = set(zip(df_hese12["run"], df_hese12["event"]))

only_in_topo   = topo_set - hese12_set
only_in_hese12 = hese12_set - topo_set

print(f"Events in topology files but not in HESE12.hdf5: {len(only_in_topo)}")
print(f"Events in HESE12.hdf5 but not in topology files: {len(only_in_hese12)}")

if not only_in_topo and not only_in_hese12:
    print("\nTopology files add up exactly to HESE12.hdf5.")
else:
    if only_in_topo:
        extra = df_topo[df_topo.apply(lambda r: (r.run, r.event) in only_in_topo, axis=1)]
        print("\nExtra events in topology files:")
        display(extra[["topology", "run", "event", "reco_energy"]].reset_index(drop=True))
    if only_in_hese12:
        missing = df_hese12[df_hese12.apply(lambda r: (r.run, r.event) in only_in_hese12, axis=1)]
        print("\nEvents missing from topology files:")
        display(missing[["run", "event", "reco_energy"]].reset_index(drop=True))

Events in topology files but not in HESE12.hdf5: 0
Events in HESE12.hdf5 but not in topology files: 0

Topology files add up exactly to HESE12.hdf5.


## 7. Compare EvtGen (after cut) with HESE12

In [61]:
hese12_set = set(zip(df_hese12["run"], df_hese12["event"]))

df_evtgen_cut["in_hese12"] = [
    (r, e) in hese12_set
    for r, e in zip(df_evtgen_cut["run"], df_evtgen_cut["event"])
]

n_matched = df_evtgen_cut["in_hese12"].sum()
print(f"EvtGen events (after cut) also in HESE12: {n_matched} / {len(df_evtgen_cut)}")

EvtGen events (after cut) also in HESE12: 83 / 102


In [62]:
# Summary per dataset
summary = df_evtgen_cut.groupby("dataset")["in_hese12"].agg(
    n_events="count", n_in_hese12="sum"
)
summary["fraction"] = summary["n_in_hese12"] / summary["n_events"]
print(summary.to_string())

           n_events  n_in_hese12  fraction
dataset                                   
IC79_2010         3            3  1.000000
IC86_2011        12           11  0.916667
IC86_2012         4            3  0.750000
IC86_2013        12           10  0.833333
IC86_2014        11           10  0.909091
IC86_2015         7            6  0.857143
IC86_2017        12           12  1.000000
IC86_2018         7            7  1.000000
IC86_2019         8            8  1.000000
IC86_2020         7            6  0.857143
IC86_2021         7            7  1.000000
IC86_2022        12            0  0.000000


In [56]:
# Events in EvtGen (after cut) but NOT in HESE12
df_not_in_hese12 = (
    df_evtgen_cut[~df_evtgen_cut["in_hese12"]]
    [["dataset", "run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"Events in EvtGen (after cut) but missing from HESE12: {len(df_not_in_hese12)}")
df_not_in_hese12

Events in EvtGen (after cut) but missing from HESE12: 19


,dataset,run,event,reco_energy
0,IC86_2011,119311,430943,3.269002e+05
1,IC86_2012,121947,7181486,7.449678e+04
2,IC86_2013,123770,442256,4.169044e+06
3,IC86_2013,122604,17469985,2.250298e+05
4,IC86_2014,125826,470241,4.164990e+05
5,IC86_2015,127751,927145,2.326454e+05
6,IC86_2020,134994,1103075,3.565653e+05
7,IC86_2022,137845,43811235,9.899828e+04
8,IC86_2022,138125,11333473,1.800607e+05
9,IC86_2022,138069,72184188,6.445069e+04


In [57]:
# Events in HESE12 but NOT in any EvtGen file (after cut)
evtgen_cut_set = set(zip(df_evtgen_cut["run"], df_evtgen_cut["event"]))
df_hese12["in_evtgen"] = [
    (r, e) in evtgen_cut_set
    for r, e in zip(df_hese12["run"], df_hese12["event"])
]

df_not_in_evtgen = (
    df_hese12[~df_hese12["in_evtgen"]]
    [["run", "event", "reco_energy"]]
    .reset_index(drop=True)
)
print(f"HESE12 events not found in EvtGen (after cut): {len(df_not_in_evtgen)}")
df_not_in_evtgen

HESE12 events not found in EvtGen (after cut): 14


,run,event,reco_energy
0,129253,4711217,259569.685977
1,128619,66564632,151213.222561
2,128809,57127218,285036.876707
3,129316,30230768,89722.762094
4,128695,52259779,74954.974283
5,129497,30635325,151115.401872
6,128224,10435404,726453.168815
7,128973,69101495,180806.123006
8,125914,75630389,70976.904502
9,128582,62891970,137869.759363
